In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rjmanoj/credit-card-customer-churn-prediction/Churn_Modelling.csv


In [44]:
df=pd.read_csv("/kaggle/input/datasets/rjmanoj/credit-card-customer-churn-prediction/Churn_Modelling.csv")

In [45]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [47]:
df.duplicated().sum()

np.int64(0)

In [48]:
df=df.drop(columns=['RowNumber','CustomerId','Surname'])

In [49]:
df['Exited'].value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

In [50]:
df['Geography'].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [51]:
df.columns = df.columns.str.strip()
print(df.columns)

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='object')


In [52]:
# # pd.get_dummies(df.columns=['Geography','Gender'],drop_first=True)
# pd.get_dummies(df.columns=['Geography','Gender'],drop_first=True)
df=pd.get_dummies(df, columns=['Geography','Gender'],drop_first=True)






In [53]:
df.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False
1,608,41,1,83807.86,1,0,1,112542.58,0,False,True,False
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False
3,699,39,1,0.00,2,0,0,93826.63,0,False,False,False
4,850,43,2,125510.82,1,1,1,79084.10,0,False,True,False


In [54]:
df=df.replace({True:1,False:0})

/tmp/ipykernel_58/3566451038.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df=df.replace({True:1,False:0})


In [55]:
df

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,0,0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0,1,0
2,502,42,8,159660.80,3,1,0,113931.57,1,0,0,0
3,699,39,1,0.00,2,0,0,93826.63,0,0,0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0.00,2,1,0,96270.64,0,0,0,1
9996,516,35,10,57369.61,1,1,1,101699.77,0,0,0,1
9997,709,36,7,0.00,1,0,1,42085.58,1,0,0,0
9998,772,42,3,75075.31,2,1,0,92888.52,1,1,0,1


In [56]:
from sklearn.model_selection import train_test_split


In [57]:
X=df.drop(columns=['Exited'])

y=df['Exited']

In [58]:
X_train,X_test,y_train,y_test=train_test_split(X,y ,test_size=0.2,random_state=45)

In [60]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

In [61]:
X_trained_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.fit_transform(X_test)

In [62]:
X_trained_scaled

array([[-0.54314547,  2.0207228 , -1.73524323, ..., -0.57812007,
        -0.57330877, -1.10331088],
       [-0.49161743, -0.66001103,  0.68725374, ...,  1.72974448,
        -0.57330877,  0.90636285],
       [ 1.01300132, -0.46853004, -0.00488825, ..., -0.57812007,
        -0.57330877,  0.90636285],
       ...,
       [ 0.90994524, -1.23445399, -0.69703024, ..., -0.57812007,
        -0.57330877, -1.10331088],
       [ 0.37405363, -0.08556807, -1.73524323, ..., -0.57812007,
        -0.57330877,  0.90636285],
       [ 0.24008073,  0.29739391, -0.35095925, ..., -0.57812007,
        -0.57330877, -1.10331088]])

In [63]:
import tensorflow
from tensorflow import keras 
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

In [78]:
model=Sequential()
model.add(Dense(3,activation='sigmoid',input_dim=11))
model.add(Dense(1,activation='sigmoid'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [66]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 3)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40 (160.00 B)

 Trainable params: 40 (160.00 B)

 Non-trainable params: 0 (0.00 B)

In [67]:
model.compile(loss='binary_crossentropy',optimizer='Adam')

In [69]:
model.fit(X_trained_scaled,y_train,epochs=10)

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5162
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4752
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4593
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4492
Epoch 5/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4417
Epoch 6/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4358
Epoch 7/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4315
Epoch 8/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4283
Epoch 9/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4259
Epoch 10/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4241


In [70]:
model.layers[0].get_weights()

[array([[-0.24705416, -0.17115356,  0.3750455 ],
        [ 0.31213412, -1.0876602 , -1.8770021 ],
        [ 0.19244419,  0.11579366,  0.10940973],
        [ 0.30104744, -0.2893623 , -0.14257027],
        [ 0.14339922,  0.09509054,  0.1605251 ],
        [-0.3506086 , -0.09204371, -0.02531312],
        [-0.92248785,  0.3669154 ,  0.58231485],
        [ 0.2896399 , -0.04928455,  0.05101264],
        [ 1.0920678 , -0.44734654, -0.29428038],
        [-0.13346446,  0.34473515, -0.34608483],
        [-0.5593325 ,  0.5090693 ,  0.08297683]], dtype=float32),
 array([-0.19752252,  0.24830957,  0.40561485], dtype=float32)]

In [71]:
model.layers[2].get_weights()

IndexError: list index out of range

In [72]:
model.layers[1].get_weights()

[array([[ 0.60497606],
        [-1.5714633 ],
        [-1.4370444 ]], dtype=float32),
 array([-0.1973467], dtype=float32)]

In [74]:
y_log=model.predict(X_test_scaled)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [75]:
y_pred=np.where(y_log>0.5,1,0)

In [76]:
from sklearn.metrics import accuracy_score

In [77]:
accuracy_score(y_test,y_pred)

0.816

In [79]:
#for imporve the accuracy  epoces increse,layer no. of noed increse , layer increase,

In [83]:
model=Sequential()
model.add(Dense(11,activation='relu',input_dim=11))
model.add(Dense(11,activation='relu'))
model.add(Dense(1,activation='sigmoid'))


In [84]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 11)             │           132 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 11)             │           132 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 276 (1.08 KB)

 Trainable params: 276 (1.08 KB)

 Non-trainable params: 0 (0.00 B)

In [86]:
# model.compile(loss='binary_crossentropy',optimizer='Adam',mertics=['accuracy'])
model.compile(
    loss='binary_crossentropy',
    optimizer='Adam',
    metrics=['accuracy']
)

In [87]:
model.fit(X_trained_scaled,y_train,epochs=100,validation_split=0.2)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7884 - loss: 0.4909 - val_accuracy: 0.7987 - val_loss: 0.4587
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7986 - loss: 0.4469 - val_accuracy: 0.8012 - val_loss: 0.4442
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8025 - loss: 0.4347 - val_accuracy: 0.8081 - val_loss: 0.4367
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8044 - loss: 0.4273 - val_accuracy: 0.8112 - val_loss: 0.4318
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8078 - loss: 0.4208 - val_accuracy: 0.8169 - val_loss: 0.4265
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8122 - loss: 0.4142 - val_accuracy: 0.8213 - val_loss: 0.4152
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8211 - loss: 0.4042 - val_accuracy: 0.8344 - val_loss: 0.4034
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8339 - loss: 0.3914 - val_accu

In [88]:
model.layers[1].get_weights()

[array([[ 0.01198507,  0.3851113 ,  0.7404325 ,  0.35521922,  0.33537236,
          0.2003122 ,  0.57538736,  0.15389174, -0.23076987,  0.10849681,
         -0.05787828],
        [ 0.02892635, -0.3387751 , -0.37723026, -0.07123832,  0.16275088,
          0.52065897, -0.16742723, -0.4586481 , -0.40306833, -0.91756463,
          0.25909454],
        [-0.4504144 , -0.0932314 ,  0.34347314, -0.06332433, -0.5391251 ,
          0.21426997, -0.50792843,  0.3457536 ,  0.10558967,  0.790043  ,
         -0.4445883 ],
        [-0.29009718,  0.18423222, -0.71489114, -0.15395139, -0.12848806,
         -0.932165  , -0.14514402, -0.21725322, -0.30052534,  0.4390719 ,
         -0.05003732],
        [-0.30853108, -0.20738655, -0.02275017,  0.54650486, -0.29633123,
          0.1166122 ,  0.42292303,  0.56918454,  0.47008556, -0.47334874,
         -0.04236709],
        [-0.3649769 ,  0.62819064,  0.39448887, -0.0140581 ,  0.42642477,
         -0.04252805, -0.23561813,  0.5032191 ,  0.23026836, -0.2532487

In [91]:
y_log=model.predict(X_test_scaled)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [92]:
y_pred=np.where(y_log>0.5,1,0)

In [93]:
from sklearn.metrics import accuracy_score

In [94]:
accuracy_score(y_test,y_pred)

0.86